# Extracao deterministica de entidades

Este notebook processa os posts em `sentinel_replica_jsons_cleaned_eval/` sem LLM. Indicadores estruturados sao extraidos por regex e entidades contextuais sao extraidas por NER local com regras deterministicas.

In [1]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

INPUT_DIR = Path("../sentinel_replica_jsons_cleaned_eval")
OUTPUT_DIR = Path("../sentinel_replica_jsons_deterministic")
OUTPUT_DIR.mkdir(exist_ok=True)

JSON_FILES = sorted(INPUT_DIR.glob("*.json"))
len(JSON_FILES), JSON_FILES[:3]

/home/ander/mestrado/Analise-Database-Cybersec/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(9,
 [PosixPath('../sentinel_replica_jsons_cleaned_eval/HackingBlogsGroup.json'),
  PosixPath('../sentinel_replica_jsons_cleaned_eval/PHOfficial.json'),
  PosixPath('../sentinel_replica_jsons_cleaned_eval/WokeIntelDrops.json')])

In [2]:
import ipaddress
import re
from urllib.parse import urlsplit

SEVERITY_LABELS = ("informational", "low", "medium", "high", "critical")
CONFIDENCE_LABELS = ("low", "medium", "high")

THREAT_LABEL_KEYWORDS = [
    ("ransomware", {"ransomware", "ransom", "encryptor", "double extortion"}),
    ("phishing", {"phishing", "smishing", "vishing", "spoofed login", "credential harvesting"}),
    ("credential_theft", {"credential harvesting", "credential stuffing", "password spraying", "session hijack", "infostealer"}),
    ("fraud_or_scam", {"scam", "fraud", "fake giveaway", "crypto drain", "drainer"}),
    ("vulnerability_or_exploit", {"cve-", "vulnerability", "exploit", "exploitation", "zero-day", "0day", "patch", "rce", "privilege escalation"}),
    ("data_breach_or_leak", {"breach", "data leak", "leaked", "exposed", "database dump", "stolen data", "stolen api", "stolen apis", "api stolen", "apis stolen", "source code stolen"}),
    ("account_takeover", {"account takeover", "ato", "hijacked account", "hacked account", "hacked email", "hacked mail", "hacked whatsapp account", "hacked phone", "hacked phones", "hacked android", "hijacked phone", "facebook was hacked", "instagram was hacked", "account back", "hacked id"}),
    ("ddos_or_disruption", {"ddos", "denial of service", "outage", "disruption"}),
    ("supply_chain_compromise", {"dependency confusion", "malicious package", "malicious update", "npm package", "pypi package"}),
    ("insider_threat", {"insider threat", "malicious insider"}),
    ("botnet_or_c2", {"botnet", "command and control", "c2", "c&c"}),
    ("initial_access_activity", {"initial access", "access broker", "vpn access", "rdp access"}),
    ("reconnaissance", {"reconnaissance", "recon", "scanning", "enumeration", "footprinting", "subdomains", "whois"}),
    ("lateral_movement", {"lateral movement", "pass-the-hash"}),
    ("exfiltration", {"exfiltration", "exfiltrate", "stolen files"}),
    ("wiper_or_destruction", {"wiper", "destructive malware", "data destruction"}),
    ("malware", {"malware", "trojan", "stealer", "loader", "backdoor", "rat", "keylogger", "virus", "spyware"}),
]

CYBER_RELEVANCE_KEYWORDS = {
    "attack", "botnet", "breach", "cve", "cyber", "exploit", "exploitation", "hack", "hacked", "hacker",
    "ioc", "malware", "phishing", "ransomware", "security", "threat", "virus", "vulnerability",
}

SEVERITY_KEYWORDS = [
    ("critical", {"critical severity", "critical vulnerability", "actively exploited", "zero-day", "0day", "rce"}),
    ("high", {"high severity", "data breach", "ransomware", "leaked", "exposed", "credential", "exploit"}),
    ("medium", {"medium severity", "phishing", "malware", "scam"}),
]

TARGET_HINTS = [
    ("critical_infrastructure", {"hospital", "healthcare", "energy", "power grid", "water utility", "telecom", "transport"}),
    ("government", {"government", "ministry", "agency", "federal", "municipal", "police", "court"}),
    ("company", {"company", "enterprise", "vendor", "customers", "organization", "firm"}),
    ("individual", {"users", "consumers", "customers", "individuals", "people"}),
]

ACTOR_TYPE_HINTS = [
    ("state", {"apt", "nation-state", "state-sponsored", "government-backed"}),
    ("hacktivist", {"hacktivist", "anonymous"}),
    ("insider", {"insider"}),
    ("criminal", {"ransomware gang", "cybercriminal", "criminal group", "fraudster"}),
]

EDUCATIONAL_REFERENCE_KEYWORDS = {
    "academy", "book", "bootcamp", "cheat sheet", "cheatsheet", "course", "ctf", "example",
    "examples", "guide", "how to", "how-to", "learn", "learning", "notes", "reference",
    "report", "research", "roadmap", "tutorial", "walkthrough", "workshop",
}

TOOL_REFERENCE_KEYWORDS = {
    "automation tool", "checker", "cheatsheet", "command", "commands", "framework", "github",
    "library", "metasploit", "module", "modules", "open source", "osint tool", "poc",
    "researchers", "scanner", "script", "search by keyword", "tool", "toolkit",
}

QUESTION_HELP_KEYWORDS = {
    "any idea", "any one know", "anyone know", "any proof", "can someone", "could someone",
    "could you provide", "do you trust", "guidance regarding", "help me", "how can i",
    "how do i", "how do we", "i need", "is there a way", "please who", "what to write",
}

OPERATIONAL_THREAT_KEYWORDS = {
    "actively exploited", "attack campaign", "breached", "breach of", "campaign", "compromised",
    "exfiltrated", "exploited in the wild", "exploited vulnerabilities", "exploited vulnerability",
    "gain access", "got hacked", "hacked account", "hit by", "infected", "intercepted",
    "leaked data", "ransomware attack", "ransomware was found", "stole", "stolen", "targeted",
    "under the hacker", "victim", "was hacked", "were hacked", "zero-day exploited",
}

REFERENCE_THREAT_LABELS = {"vulnerability_or_exploit", "reconnaissance", "malware", "botnet_or_c2"}
SENSITIVE_OPERATIONAL_LABELS = {
    "credential_theft", "data_breach_or_leak", "ddos_or_disruption", "phishing", "ransomware",
    "supply_chain_compromise", "account_takeover", "exfiltration", "wiper_or_destruction",
}

LABEL_PRIORITY = [
    "ransomware", "supply_chain_compromise", "data_breach_or_leak", "vulnerability_or_exploit",
    "account_takeover", "phishing", "credential_theft", "malware", "botnet_or_c2",
    "ddos_or_disruption", "initial_access_activity", "exfiltration", "wiper_or_destruction",
    "insider_threat", "lateral_movement", "reconnaissance", "fraud_or_scam",
]

NEGATED_LABEL_PATTERNS = {
    "data_breach_or_leak": [
        r"\bchannels?\s+related\s+to\b.{0,80}\b(?:data\s+)?leaks?\b",
    ],
    "fraud_or_scam": [
        r"\b(?:not|no|isn[' ]?t|ain[' ]?t|without)\s+(?:a\s+)?(?:scam|fraud)\b",
        r"\bit\s+s\s+not\s+(?:scam|fraud)\b",
    ],
    "phishing": [r"\b(?:not|no|isn[' ]?t|ain[' ]?t|without)\s+(?:a\s+)?phishing\b"],
    "credential_theft": [r"\bcookie\s+delete\b", r"\bdelete\s+(?:the\s+)?cookies?\b"],
    "malware": [r"\bboot\s+loader\b", r"\bjob\s+in\s+backdoor\b", r"\bbackdoor\s+jobs?\b", r"\bbackdoor\s+now\b"],
    "supply_chain_compromise": [
        r"\bstrengthening\s+the\s+supply\s+chain\b",
        r"\bsupply\s+chain\s+attacks?\b.{0,80}\b(?:monitoring|asm|attack\s+surface\s+management)\b",
        r"\b(?:curriculum|course|training|skill\s+area)\b.{0,160}\bsupply\s+chain\s+compromises?\b",
    ],
    "vulnerability_or_exploit": [
        r"\bvulnerability\s+management\b.{0,120}\b(?:opportunities|internship|fresher|referrals|roles?)\b",
        r"\b(?:don\s+t|do\s+not|don\s+’\s+t)\s+need\s+exploit\s+code\b",
        r"\bsecurity\s+and\s+privacy\s+vision\b",
    ],
}

REGEX_LABEL_PATTERNS = {
    "account_takeover": [
        r"\b(?:facebook|instagram|whatsapp|telegram|bgmi|social\s+media)\b.{0,60}\b(?:was|has\s+been|got)\s+hacked\b",
        r"\b(?:email|e-mail|mail)\b.{0,40}\b(?:was|has\s+been|got)?\s*hacked\b",
        r"\b(?:recover|bring\s+back|get\s+back)\b.{0,80}\b(?:account|id)\b",
        r"\b(?:account|id)\b.{0,40}\b(?:was|has\s+been|got)\s+hacked\b",
    ],
    "data_breach_or_leak": [
        r"\b(?:api|apis|database|credentials?|source\s+code|tokens?)\b.{0,40}\b(?:stolen|leaked|exposed)\b",
        r"\b(?:stolen|leaked|exposed)\b.{0,40}\b(?:api|apis|database|credentials?|source\s+code|tokens?)\b",
        r"\b(?:executed|planned|coordinated)\s+leak\b",
    ],
    "reconnaissance": [
        r"\btool\s+that\s+provides\s+information\s+about\s+a\s+website\b",
        r"\b(?:subdomains?|whois|footprinting)\b.{0,80}\b(?:website|domain|osint|recon)\b",
    ],
    "credential_theft": [
        r"\b(?:password|credentials?|token|cookie|session)\b.{0,50}\b(?:stolen|leaked|exposed|hacked|harvested|dumped|changed)\b",
        r"\b(?:stolen|leaked|exposed|hacked|harvested|dumped|changed)\b.{0,50}\b(?:password|credentials?|token|cookie|session)\b",
        r"\b(?:login|email)\s+and\s+password\b",
    ],
    "supply_chain_compromise": [
        r"\bsupply\s+chain\s+(?:attack|compromise|incident)\b",
        r"\b(?:compromised|malicious|hacked)\s+(?:dependency|dependencies|package|library|update|version|versions)\b",
        r"\b(?:dependency|dependencies|package|library|update|version|versions)\b.{0,60}\b(?:compromised|malicious|hacked|steals?|delivers?|infects?)\b",
    ],
    "vulnerability_or_exploit": [
        r"\bpoc\s+available\b",
        r"\bproof\s+of\s+concept\s+available\b",
    ],
}

NER_TTP_LABELS = {
    "phishing": "phishing",
    "spear phishing": "phishing",
    "credential stuffing": "credential_theft",
    "password spraying": "credential_theft",
    "data exfiltration": "exfiltration",
    "lateral movement": "lateral_movement",
    "privilege escalation": "vulnerability_or_exploit",
    "remote code execution": "vulnerability_or_exploit",
    "sql injection": "vulnerability_or_exploit",
    "cross-site scripting": "vulnerability_or_exploit",
    "brute force": "initial_access_activity",
}

In [3]:
SPACY_MODEL = "en_core_web_sm"
NER_N_PROCESS = max(1, os.cpu_count() or 1)
NER_BATCH_SIZE = 256

try:
    import spacy
    from spacy.language import Language
except ImportError as exc:
    raise ImportError(
        "Instale spaCy para executar o NER local: `uv add spacy` "
        "e, opcionalmente, `python -m spacy download en_core_web_sm`."
    ) from exc


def _load_ner_pipeline(model_name: str = SPACY_MODEL) -> Language:
    try:
        nlp = spacy.load(model_name)
        source = model_name
    except OSError:
        nlp = spacy.blank("en")
        nlp.add_pipe("sentencizer")
        source = "spacy.blank(en)+entity_ruler"

    if "entity_ruler" not in nlp.pipe_names:
        if "ner" in nlp.pipe_names:
            ruler = nlp.add_pipe("entity_ruler", before="ner")
        else:
            ruler = nlp.add_pipe("entity_ruler")
    else:
        ruler = nlp.get_pipe("entity_ruler")

    ruler.add_patterns(NER_PATTERNS)
    print(f"NER pipeline carregado: {source}")
    return nlp


NER_PATTERNS = [
    {"label": "MALWARE", "pattern": name}
    for name in [
        "RedLine", "Raccoon Stealer", "Lumma", "Lumma Stealer", "AsyncRAT", "Remcos",
        "Agent Tesla", "QakBot", "Emotet", "TrickBot", "LockBit", "BlackCat", "ALPHV",
        "Clop", "Akira", "Rhysida", "Mirai", "XLoader", "FormBook", "Vidar", "Meduza Stealer",
    ]
] + [
    {"label": "TOOL", "pattern": name}
    for name in [
        "Cobalt Strike", "Metasploit", "Mimikatz", "Nmap", "Burp Suite", "Sliver", "BloodHound",
        "AnyDesk", "TeamViewer", "Rclone", "PowerShell", "PsExec",
    ]
] + [
    {"label": "THREAT_ACTOR", "pattern": name}
    for name in [
        "APT28", "APT29", "Lazarus", "Sandworm", "FIN7", "Scattered Spider", "ShinyHunters",
        "LockBit", "Clop", "BlackCat", "ALPHV", "Anonymous Sudan", "KillNet",
    ]
] + [
    {"label": "PRODUCT", "pattern": name}
    for name in [
        "Windows", "Linux", "macOS", "Android", "iOS", "Chrome", "Firefox", "Safari",
        "WordPress", "SharePoint", "Exchange", "FortiGate", "Cisco IOS", "Apache", "OpenSSH",
        "Citrix NetScaler", "Ivanti Connect Secure", "VMware ESXi", "Google Play",
    ]
] + [
    {"label": "SECTOR", "pattern": name}
    for name in [
        "healthcare", "financial services", "government", "education", "energy", "telecom",
        "retail", "technology sector", "technology industry", "technology companies",
        "critical infrastructure", "transportation",
        "defense sector", "defense industry", "defense contractor", "defense contractors",
        "Department of Defense", "Ministry of Defense",
    ]
] + [
    {"label": "TTP", "pattern": name}
    for name in [
        "phishing", "spear phishing", "credential stuffing", "lateral movement", "privilege escalation",
        "data exfiltration", "command and control", "remote code execution", "SQL injection",
        "cross-site scripting", "brute force", "password spraying",
    ]
] + [
    {"label": "VULNERABILITY", "pattern": name}
    for name in [
        "buffer overflow", "heap-based buffer overflow", "authentication bypass",
        "authorization bypass", "path traversal", "directory traversal", "deserialization",
        "server-side request forgery", "SSRF", "server-side template injection", "SSTI",
        "cross-site request forgery", "CSRF", "XML external entity", "XXE",
        "local file inclusion", "LFI", "remote file inclusion", "RFI", "zero-day vulnerability",
    ]
] + [
    {"label": "COUNTRY", "pattern": name}
    for name in [
        "United States", "U.S.", "USA", "United Kingdom", "U.K.", "Russia", "China",
        "North Korea", "Iran", "Israel", "India", "Brazil", "Canada", "Australia",
        "Ukraine", "Germany", "France", "Japan", "Bangladesh", "Chad",
    ]
]

NER = _load_ner_pipeline()

NER pipeline carregado: spacy.blank(en)+entity_ruler


In [4]:
URL_RE = re.compile(r"https?://[^\s<>\")\]]+", re.I)
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.I)
IP_RE = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
CVE_RE = re.compile(r"\bCVE-\d{4}-\d{4,7}\b", re.I)
HASH_RE = re.compile(r"\b[a-fA-F0-9]{32}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{64}\b")
ETH_RE = re.compile(r"\b0x[a-fA-F0-9]{40}\b")
BTC_RE = re.compile(r"\b(?:bc1|[13])[a-zA-HJ-NP-Z0-9]{25,62}\b")
HANDLE_RE = re.compile(r"(?<![\w@])@[A-Za-z0-9_]{3,32}\b")

DOMAIN_RE = re.compile(
    r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}\b",
    re.I,
)
DEFANGED_DOMAIN_RE = re.compile(
    r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?(?:\.|\[\.\]|\(\.\)))+[a-z]{2,}\b",
    re.I,
)

_SOCIAL_DOMAINS = {
    "linkedin.com", "lnkd.in", "twitter.com", "x.com", "t.me", "telegram.me",
    "youtube.com", "youtu.be", "facebook.com", "instagram.com", "tiktok.com",
    "github.com", "github.io", "udemy.com", "coursera.org", "udacity.com",
    "medium.com", "whatsapp.com",
}

_NON_IOC_DOMAINS = _SOCIAL_DOMAINS | {
    "blog.cloudflare.com", "drive.google.com", "gitlab.com", "gist.github.com", "google.com",
    "professionalhackers.in", "telemodsapk.com",
}

_COMMON_TLDS = {
    "app", "br", "biz", "cloud", "cn", "co", "com", "dev", "edu", "gov", "info",
    "io", "ir", "me", "mil", "net", "news", "org", "pro", "ru", "run", "site",
    "tech", "to", "top", "uk", "us", "xyz",
}

_ENTITY_TYPE_TO_OUTPUT = {
    "ip": "ip_address",
    "url": "url",
    "domain": "domain",
    "email": "email",
    "cve": "cve",
    "hash": "hash",
    "wallet": "wallet",
    "handle": "handle",
    "actor": "threat_actor",
    "malware": "malware",
    "tool": "tool",
    "product": "product",
    "vuln": "vulnerability",
    "victim": "victim",
    "sector": "sector",
    "country": "country",
    "ttp": "ttp",
}

_IOC_ENTITY_TYPES = {
    "ip_address", "domain", "url", "hash", "email", "wallet", "handle", "cve"
}


def _clean_regex_value(value: str) -> str:
    return value.strip().rstrip(".,;:)]}")


def _refang_domain(value: str) -> str:
    return value.replace("[.]", ".").replace("(.)", ".")


def _strip_www(domain: str) -> str:
    domain = domain.lower()
    return domain[4:] if domain.startswith("www.") else domain


def _url_domain(url: str) -> str:
    try:
        return _strip_www(urlsplit(url).netloc)
    except ValueError:
        return ""


def _looks_like_domain(value: str) -> bool:
    domain = _strip_www(_refang_domain(value).rstrip("."))
    parts = domain.rsplit(".", 1)
    return len(parts) == 2 and parts[1] in _COMMON_TLDS


def _valid_ip_address(value: str) -> bool:
    try:
        address = ipaddress.ip_address(value)
    except ValueError:
        return False
    return not any([
        address.is_private,
        address.is_loopback,
        address.is_multicast,
        address.is_reserved,
        address.is_unspecified,
    ])


def _looks_like_version_number(text: str, value: str) -> bool:
    return re.search(rf"\b(?:version|before\s+version|v)\s+{re.escape(value)}\b", text, re.I) is not None


def _valid_wallet(value: str) -> bool:
    if value.lower().startswith("0x"):
        return True
    if value.lower().startswith("bc1"):
        return True
    if re.fullmatch(r"[13][a-fA-F0-9]{25,62}", value):
        return False
    return True


def _valid_handle(value: str) -> bool:
    return value.lower() not in {"@echo"}


def _valid_source_url(url: str) -> str:
    url = _clean_regex_value(url)
    if not url.startswith("http"):
        return ""
    netloc = _url_domain(url)
    if not netloc:
        return ""
    if any(netloc == d or netloc.endswith("." + d) for d in _SOCIAL_DOMAINS):
        return ""
    return url


def _entity(entity_type: str, value: str, *, source: str = "regex") -> dict[str, str]:
    output_type = _ENTITY_TYPE_TO_OUTPUT.get(entity_type, entity_type)
    item = {"type": output_type, "value": value, "source": source}
    if output_type == "url":
        domain = _url_domain(value)
        if domain:
            item["domain"] = domain
    return item


def extract_regex_entities(text: str) -> list[dict[str, str]]:
    urls = [_clean_regex_value(url) for url in URL_RE.findall(text)]

    # Remove URLs antes de capturar domínios soltos, para não duplicar hosts de links.
    text_no_urls = URL_RE.sub(" ", text)

    entities: list[dict[str, str]] = []

    def add(entity_type: str, values: list[str], validator=None) -> None:
        seen = set()
        for value in values:
            value = _clean_regex_value(value)
            if validator and not validator(value):
                continue
            key = (entity_type, value.lower())
            if value and key not in seen:
                seen.add(key)
                entities.append(_entity(entity_type, value))

    add("url", urls)
    add("email", EMAIL_RE.findall(text))
    add("ip", IP_RE.findall(text_no_urls), lambda value: _valid_ip_address(value) and not _looks_like_version_number(text_no_urls, value))
    add("cve", CVE_RE.findall(text))
    add("hash", HASH_RE.findall(text_no_urls))
    add("wallet", ETH_RE.findall(text_no_urls) + BTC_RE.findall(text_no_urls), _valid_wallet)
    add("handle", HANDLE_RE.findall(text_no_urls), _valid_handle)
    add("domain", DOMAIN_RE.findall(text_no_urls), _looks_like_domain)
    add("domain", [_refang_domain(value) for value in DEFANGED_DOMAIN_RE.findall(text_no_urls)], _looks_like_domain)

    return entities


def extract_primary_source_url(text: str) -> str:
    for url in URL_RE.findall(text):
        valid = _valid_source_url(url)
        if valid:
            return valid
    return ""


def _is_social_domain(domain: str) -> bool:
    domain = _strip_www(domain)
    return any(domain == d or domain.endswith("." + d) for d in _SOCIAL_DOMAINS)


def _is_non_ioc_domain(domain: str) -> bool:
    domain = _strip_www(domain)
    return any(domain == d or domain.endswith("." + d) for d in _NON_IOC_DOMAINS)


def _is_likely_ioc_entity(entity: dict[str, str], primary_source_url: str) -> bool:
    entity_type = entity.get("type", "")
    value = entity.get("value", "")
    if entity_type not in _IOC_ENTITY_TYPES:
        return False
    if entity_type in {"cve", "hash", "ip_address", "email", "wallet"}:
        return True
    if entity_type == "url":
        domain = entity.get("domain", "") or _url_domain(value)
        return bool(domain) and not _is_non_ioc_domain(domain) and value != primary_source_url
    if entity_type == "domain":
        primary_domain = _url_domain(primary_source_url) if primary_source_url else ""
        return bool(value) and not _is_non_ioc_domain(value) and value.lower() != primary_domain
    return False


_BAD_CONTEXT_VALUES = {
    "n/a", "na", "none", "null", "unknown", "not specified", "not applicable",
    "generic", "informational", "low", "medium", "high", "critical",
    "it", "i'm", "i'd", "previously", "and tor",
    "i", "l", "m", "h", "c", "u", "i|l|m|h|c", "l|m|h", "ind|org|gov|ci|u",
}

_GENERIC_VICTIM_TERMS = {
    "advisory", "affected", "article", "blockchain", "bootcamp", "community", "consumers",
    "account", "accounts", "cctv", "facebook", "generic", "instagram", "owners",
    "plcs", "rdp", "readers", "twitter", "users", "vulnerability", "whatsapp", "workshop",
}

_BAD_VICTIM_VALUES = {
    "ai", "and tor", "i'd", "i'm", "india's", "ip address", "it", "obama's",
    "previously", "read", "sqli", "these", "u.s", "vpn", "who", "wi-fi",
}

_PRODUCT_HINTS = {
    "android", "apache", "app", "chrome", "cups", "driver", "firefox", "ios",
    "kernel", "linux", "macos", "plugin", "router", "safari", "server", "sharepoint",
    "solana web3.js", "tools", "windows", "wordpress",
}

_PROMOTIONAL_NAMES = {
    "hackingblogs", "hackingblogs.com", "api-hacking bootcamp", "hackingblogs community",
}

_GOV_HINTS = {
    "agency", "court", "department", "federal", "government", "ministry", "national",
    "police", "treasury",
}


def _clean_context_name(value: str) -> str:
    value = re.sub(r"\s+", " ", value.strip())
    return value.strip(" .,;:-")


def _valid_context_name(value: str) -> bool:
    normalized = _clean_context_name(value)
    lowered = normalized.lower()
    if not normalized or lowered in _BAD_CONTEXT_VALUES:
        return False
    if "|" in normalized or "{" in normalized or "}" in normalized:
        return False
    if re.fullmatch(r"[a-z](?:\|[a-z])+", lowered):
        return False
    if CVE_RE.search(normalized):
        return False
    if normalized.islower() and len(normalized) <= 3:
        return False
    if len(normalized) > 96 or len(normalized.split()) > 8:
        return False
    return True


def _looks_promotional_or_generic(name: str) -> bool:
    lowered = name.lower()
    if lowered in _PROMOTIONAL_NAMES or any(p in lowered for p in _PROMOTIONAL_NAMES):
        return True
    tokens = set(re.findall(r"[a-z0-9]+", lowered))
    return bool(tokens & _GENERIC_VICTIM_TERMS)


def _looks_like_product(name: str) -> bool:
    lowered = name.lower()
    if "." in lowered and " " not in lowered:
        return True
    return any(hint in lowered for hint in _PRODUCT_HINTS)


def _looks_like_malware_or_tool(name: str) -> bool:
    lowered = name.lower()
    return any(term in lowered for term in {"keylogger", "malware", "ransomware", "stealer"})


def _infer_victim_type(name: str, target_type: str) -> str:
    if target_type != "unknown":
        return target_type
    lowered = name.lower()
    if any(hint in lowered for hint in _GOV_HINTS):
        return "government"
    return "company"


def _should_keep_victim(name: str, target_type: str, threat_label: str) -> bool:
    if not _valid_context_name(name):
        return False
    if name.lower() in _BAD_VICTIM_VALUES:
        return False
    if name.endswith("'s"):
        return False
    if re.search(r"\.\s+[A-Z][a-z]+$", name):
        return False
    if any(sep in name for sep in [",", ";"]):
        return False
    if _looks_like_domain(name):
        return False
    if _looks_promotional_or_generic(name):
        return False
    if _looks_like_malware_or_tool(name):
        return False
    if threat_label == "vulnerability_or_exploit" and (target_type == "unknown" or _looks_like_product(name)):
        return False
    return True


def _add_entity_once(
    entities: list[dict[str, str]],
    entity_type: str,
    value: str,
    source: str = "ner",
    **extra: str,
) -> None:
    value = _clean_context_name(value)
    if not _valid_context_name(value):
        return
    output_type = _ENTITY_TYPE_TO_OUTPUT.get(entity_type, entity_type)
    key = (output_type, value.lower())
    if all((ent.get("type"), ent.get("value", "").lower()) != key for ent in entities):
        ent = _entity(entity_type, value, source=source)
        ent.update({k: v for k, v in extra.items() if v})
        entities.append(ent)


def _structured_name_entities(
    entities: list[dict[str, str]],
    entity_type: str,
    output_type: str,
    threat_label: str,
) -> list[dict[str, str]]:
    values: list[dict[str, str]] = []
    seen = set()
    for ent in entities:
        if ent.get("type") != entity_type:
            continue
        name = _clean_context_name(ent.get("value", ""))
        if not _should_keep_victim(name, output_type, threat_label):
            continue
        key = name.lower()
        if key not in seen:
            seen.add(key)
            values.append({"name": name, "type": _infer_victim_type(name, output_type), "source": ent.get("source", "ner")})
    return values


def _build_victims(
    victim_name: str,
    target_type: str,
    threat_label: str,
    entities: list[dict[str, str]],
) -> list[dict[str, str]]:
    victims = _structured_name_entities(entities, "victim", target_type, threat_label)
    primary_name = _clean_context_name(victim_name)
    if _should_keep_victim(primary_name, target_type, threat_label) and all(v["name"].lower() != primary_name.lower() for v in victims):
        victims.insert(0, {"name": primary_name, "type": _infer_victim_type(primary_name, target_type), "source": "ner"})
    return victims


def _sync_victims_to_entities(victims: list[dict[str, str]], entities: list[dict[str, str]]) -> None:
    for victim in victims:
        _add_entity_once(
            entities,
            "victim",
            victim.get("name", ""),
            source=victim.get("source", "ner"),
            victim_type=victim.get("type", ""),
        )


def valid_incident_date(value: str) -> str:
    value = value.strip()
    return value if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value) else ""

In [5]:
NER_LABEL_TO_ENTITY_TYPE = {
    "MALWARE": "malware",
    "TOOL": "tool",
    "THREAT_ACTOR": "actor",
    "PRODUCT": "product",
    "VULNERABILITY": "vuln",
    "SECTOR": "sector",
    "COUNTRY": "country",
    "TTP": "ttp",
}

ACTOR_CONTEXT = {"apt", "actor", "gang", "group", "hackers", "threat actor", "ransomware gang"}
VICTIM_CONTEXT = {"against", "breach", "compromised", "exposed", "hit", "impacted", "leak", "targeted", "victim"}

CONTEXTUAL_VICTIM_PATTERNS = [
    re.compile(r"\b(?:attack|breach|leak|incident|campaign)\s+(?:on|against|at|targeting)\s+([A-Z][A-Za-z0-9&.'-]*(?:\s+[A-Z][A-Za-z0-9&.'-]*){0,5})"),
    re.compile(r"\b([A-Z][A-Za-z0-9&.'-]*(?:\s+[A-Z][A-Za-z0-9&.'-]*){0,5})\s+(?:was|were|has\s+been|have\s+been|is\s+being|are\s+being)\s+(?:hacked|breached|compromised|targeted|hit|impacted|exposed)\b"),
    re.compile(r"\b(?:hacked|breached|compromised|targeted|hit|impacted)\s+([A-Z][A-Za-z0-9&.'-]*(?:\s+[A-Z][A-Za-z0-9&.'-]*){0,5})\b"),
]


def _contains_keyword(text: str, keyword: str) -> bool:
    if any(char in keyword for char in " -&"):
        return keyword in text
    return re.search(rf"\b{re.escape(keyword)}\b", text) is not None


def _matched_keywords(text: str, keywords: set[str]) -> set[str]:
    lowered = text.lower()
    return {keyword for keyword in keywords if _contains_keyword(lowered, keyword)}


def _entity_context(text: str, start: int, end: int, size: int = 90) -> str:
    return text[max(0, start - size): min(len(text), end + size)].lower()


def _infer_contextual_entity_type(label: str, context: str) -> str:
    if label in NER_LABEL_TO_ENTITY_TYPE:
        return NER_LABEL_TO_ENTITY_TYPE[label]
    if label in {"GPE", "LOC"}:
        return "country"
    if label == "PRODUCT":
        return "product"
    if label in {"ORG", "PERSON"}:
        if _matched_keywords(context, ACTOR_CONTEXT):
            return "actor"
        if _matched_keywords(context, VICTIM_CONTEXT):
            return "victim"
    return ""


def _extract_contextual_victims(text: str) -> list[str]:
    values: list[str] = []
    for pattern in CONTEXTUAL_VICTIM_PATTERNS:
        for match in pattern.finditer(text):
            name = _clean_context_name(match.group(1))
            if _should_keep_victim(name, "unknown", ""):
                values.append(name)
    return values


def extract_ner_entities_from_doc(text: str, doc: Any) -> list[dict[str, str]]:
    entities: list[dict[str, str]] = []
    for ent in doc.ents:
        value = _clean_context_name(ent.text)
        if not _valid_context_name(value) or _looks_like_domain(value):
            continue
        entity_type = _infer_contextual_entity_type(ent.label_, _entity_context(text, ent.start_char, ent.end_char))
        if entity_type:
            _add_entity_once(entities, entity_type, value, source="ner")
    for victim in _extract_contextual_victims(text):
        _add_entity_once(entities, "victim", victim, source="ner")
    return entities


def extract_ner_entities(text: str) -> list[dict[str, str]]:
    return extract_ner_entities_from_doc(text, NER(text))


def _dedupe_entities(items: list[dict[str, str]]) -> list[dict[str, str]]:
    entities: list[dict[str, str]] = []
    seen = set()
    for ent in items:
        key = (ent.get("type", ""), ent.get("value", "").lower())
        if key not in seen:
            seen.add(key)
            entities.append(ent)
    return entities


def _remove_contextual_false_positives(labels: list[str], text: str) -> list[str]:
    filtered: list[str] = []
    for label in labels:
        patterns = NEGATED_LABEL_PATTERNS.get(label, [])
        if any(re.search(pattern, text) for pattern in patterns):
            continue
        filtered.append(label)
    return filtered


def _prioritize_labels(labels: list[str], regex_entities: list[dict[str, str]]) -> list[str]:
    deduped = list(dict.fromkeys(labels))
    if any(ent.get("type") == "cve" for ent in regex_entities) and "vulnerability_or_exploit" in deduped:
        deduped = ["vulnerability_or_exploit"] + [label for label in deduped if label != "vulnerability_or_exploit"]
    priority = {label: idx for idx, label in enumerate(LABEL_PRIORITY)}
    return sorted(deduped, key=lambda label: priority.get(label, len(priority)))


def _infer_regex_pattern_labels(text: str) -> list[str]:
    labels: list[str] = []
    for label, patterns in REGEX_LABEL_PATTERNS.items():
        if any(re.search(pattern, text) for pattern in patterns):
            labels.append(label)
    return labels


def _infer_ner_ttp_labels(ner_entities: list[dict[str, str]]) -> list[str]:
    labels: list[str] = []
    for ent in ner_entities:
        if ent.get("type") != "ttp":
            continue
        label = NER_TTP_LABELS.get(ent.get("value", "").lower())
        if label:
            labels.append(label)
    return labels


def _infer_threat_labels(text: str, regex_entities: list[dict[str, str]], ner_entities: list[dict[str, str]]) -> list[str]:
    labels: list[str] = []
    for label, keywords in THREAT_LABEL_KEYWORDS:
        if _matched_keywords(text, keywords):
            labels.append(label)
    labels.extend(_infer_regex_pattern_labels(text))

    if any(ent.get("type") == "cve" for ent in regex_entities):
        labels.append("vulnerability_or_exploit")
    if any(ent.get("type") == "vulnerability" for ent in ner_entities):
        labels.append("vulnerability_or_exploit")
    if any(ent.get("type") == "malware" for ent in ner_entities):
        labels.append("malware")
    labels.extend(_infer_ner_ttp_labels(ner_entities))

    labels = _remove_contextual_false_positives(list(dict.fromkeys(labels)), text)
    return _prioritize_labels(labels, regex_entities)


def _has_operational_context(text: str) -> bool:
    return bool(_matched_keywords(text, OPERATIONAL_THREAT_KEYWORDS))


def _infer_content_type(text: str, has_operational_context: bool) -> str:
    if has_operational_context:
        return "operational_threat"
    if _matched_keywords(text, QUESTION_HELP_KEYWORDS):
        return "question_or_help"
    has_educational_terms = bool(_matched_keywords(text, EDUCATIONAL_REFERENCE_KEYWORDS))
    has_tool_terms = bool(_matched_keywords(text, TOOL_REFERENCE_KEYWORDS))
    if has_educational_terms:
        return "educational_reference"
    if has_tool_terms:
        return "tool_reference"
    return "general_cyber"


def _is_non_operational_content(content_type: str, has_operational_context: bool) -> bool:
    return content_type in {"educational_reference", "tool_reference", "question_or_help"} and not has_operational_context


def _adjust_labels_for_content(
    labels: list[str],
    content_type: str,
    has_operational_context: bool,
    regex_entities: list[dict[str, str]],
) -> list[str]:
    if not _is_non_operational_content(content_type, has_operational_context):
        return labels
    if any(ent.get("type") == "cve" for ent in regex_entities):
        return ["vulnerability_or_exploit"]
    if content_type in {"educational_reference", "tool_reference"} and "reconnaissance" in labels:
        return ["reconnaissance"] + [
            label for label in labels
            if label not in {"reconnaissance", "vulnerability_or_exploit"}
        ]
    if content_type in {"educational_reference", "tool_reference"} and "lateral_movement" in labels:
        return ["lateral_movement"] + [
            label for label in labels
            if label not in {"lateral_movement", "ransomware"}
        ]
    return labels


def _infer_target_type(text: str, entities: list[dict[str, str]]) -> str:
    for target_type, keywords in TARGET_HINTS:
        if _matched_keywords(text, keywords):
            return target_type
    if any(ent.get("type") == "sector" and ent.get("value", "").lower() in {"government", "defense"} for ent in entities):
        return "government"
    return "unknown"


def _infer_actor_type(text: str) -> str:
    for actor_type, keywords in ACTOR_TYPE_HINTS:
        if _matched_keywords(text, keywords):
            return actor_type
    return "unknown"


def _infer_severity(
    text: str,
    threat_label: str,
    ioc_types: list[str],
    content_type: str,
    has_operational_context: bool,
) -> str:
    if _is_non_operational_content(content_type, has_operational_context):
        return "informational"
    if threat_label in {"not_a_threat", "other_cyber"}:
        return "medium" if has_operational_context or ioc_types else "informational"
    for severity, keywords in SEVERITY_KEYWORDS:
        if _matched_keywords(text, keywords):
            return severity
    if threat_label in {"ransomware", "data_breach_or_leak", "supply_chain_compromise"}:
        return "high"
    if ioc_types or threat_label != "not_a_threat":
        return "medium"
    return "informational"


def _infer_confidence(
    threat_label: str,
    labels: list[str],
    ioc_types: list[str],
    ner_entities: list[dict[str, str]],
    content_type: str,
) -> str:
    if threat_label == "not_a_threat":
        return "medium"
    if content_type in {"educational_reference", "tool_reference", "question_or_help"}:
        return "high"
    if ioc_types or ner_entities:
        return "high"
    if labels:
        return "medium"
    return "low"


def _extract_incident_date(text: str) -> str:
    for value in re.findall(r"\b\d{4}-\d{2}-\d{2}\b", text):
        if valid_incident_date(value):
            return value
    return ""


def _post_text_fields(post: dict[str, Any]) -> tuple[str, str, str]:
    cleaned_text = str(
        post.get("cleaning", {}).get("normalized_message")
        or post.get("message", "")
        or ""
    )
    original_text = str(post.get("message") or cleaned_text or "")
    analysis_text = f"{original_text}\n{cleaned_text}".strip()
    return cleaned_text, original_text, analysis_text


def normalize_deterministic_result(
    post: dict[str, Any],
    ner_entities: list[dict[str, str]] | None = None,
) -> dict[str, Any]:
    cleaned_text, original_text, analysis_text = _post_text_fields(post)
    lowered_text = analysis_text.lower()

    primary_source_url = extract_primary_source_url(original_text)
    regex_entities = extract_regex_entities(original_text)
    if ner_entities is None:
        ner_entities = extract_ner_entities(analysis_text)
    entities = _dedupe_entities(regex_entities + ner_entities)

    ioc_types = sorted({
        ent["type"]
        for ent in entities
        if _is_likely_ioc_entity(ent, primary_source_url)
    })

    has_operational_context = _has_operational_context(lowered_text)
    content_type = _infer_content_type(lowered_text, has_operational_context)
    raw_inferred_labels = _infer_threat_labels(lowered_text, regex_entities, ner_entities)
    inferred_labels = _adjust_labels_for_content(
        raw_inferred_labels,
        content_type,
        has_operational_context,
        regex_entities,
    )
    inferred_labels = _prioritize_labels(inferred_labels, regex_entities)
    is_cyber_relevant = bool(
        inferred_labels
        or raw_inferred_labels
        or ioc_types
        or _matched_keywords(lowered_text, CYBER_RELEVANCE_KEYWORDS)
    )
    threat_label = inferred_labels[0] if inferred_labels else ("other_cyber" if is_cyber_relevant else "not_a_threat")
    secondary_labels = [label for label in inferred_labels[1:] if label != threat_label]
    target_type = _infer_target_type(lowered_text, entities)
    actor_type = _infer_actor_type(lowered_text)

    for ent in entities:
        if ent.get("type") == "threat_actor" and actor_type != "unknown":
            ent["actor_type"] = actor_type

    if not is_cyber_relevant:
        return {
            "cleaned_text": cleaned_text,
            "is_cyber_relevant": False,
            "threat_label": "not_a_threat",
            "secondary_labels": [],
            "content_type": content_type,
            "severity": "informational",
            "confidence": "medium",
            "target_type": "unknown",
            "ioc_present": bool(ioc_types),
            "ioc_types": ioc_types,
            "requires_human_review": False,
            "victim_name": "",
            "victims": [],
            "primary_source_url": primary_source_url,
            "incident_date": "",
            "entities": entities,
        }

    victims = _build_victims("", target_type, threat_label, entities)
    _sync_victims_to_entities(victims, entities)
    victim_name = victims[0]["name"] if victims else ""
    incident_date = _extract_incident_date(original_text)
    confidence = _infer_confidence(threat_label, inferred_labels, ioc_types, ner_entities, content_type)

    requires_review = (
        confidence == "low"
        or not victim_name and threat_label in {
            "ransomware",
            "data_breach_or_leak",
            "supply_chain_compromise",
            "account_takeover",
        }
    )

    return {
        "cleaned_text": cleaned_text,
        "is_cyber_relevant": True,
        "threat_label": threat_label,
        "secondary_labels": secondary_labels,
        "content_type": content_type,
        "severity": _infer_severity(lowered_text, threat_label, ioc_types, content_type, has_operational_context),
        "confidence": confidence,
        "target_type": target_type,
        "ioc_present": bool(ioc_types),
        "ioc_types": ioc_types,
        "requires_human_review": requires_review,
        "victim_name": victim_name,
        "victims": victims,
        "primary_source_url": primary_source_url,
        "incident_date": incident_date,
        "entities": entities,
    }

In [6]:
def parse_post_datetime(post: dict[str, Any]) -> datetime | None:
    value = post.get("datetime_utc")
    if isinstance(value, str) and value.strip():
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)
        except ValueError:
            pass

    value = post.get("timestamp")
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value, tz=timezone.utc)

    value = post.get("date")
    if isinstance(value, str) and value.strip():
        for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S"):
            try:
                return datetime.strptime(value[:19], fmt).replace(tzinfo=timezone.utc)
            except ValueError:
                continue
    return None


def ensure_temporal_fields(post: dict[str, Any]) -> dict[str, Any]:
    enriched = dict(post)
    parsed = parse_post_datetime(enriched)
    if parsed is not None:
        enriched["datetime_utc"] = parsed.isoformat()
        enriched["timestamp"] = int(parsed.timestamp())
        enriched["date"] = parsed.strftime("%Y-%m-%d")
    return enriched


records = []
for json_file in JSON_FILES:
    with json_file.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    if isinstance(payload, list):
        for item in payload:
            records.append({"source_file": json_file.name, **ensure_temporal_fields(item)})

df = pd.DataFrame(records)
display_columns = [
    column
    for column in ["source_file", "_id", "date", "datetime_utc", "timestamp", "message", "cleaning"]
    if column in df.columns
]
df[display_columns].head() if display_columns else df.head()


,source_file,_id,date,datetime_utc,timestamp,message,cleaning
0,HackingBlogsGroup.json,26,2024-07-03,2024-07-03T02:34:12+00:00,1719974052,Detail Article on Apple Internal Tool Source C...,{'normalized_message': 'detail article on appl...
1,HackingBlogsGroup.json,31,2024-07-07,2024-07-07T00:58:02+00:00,1720313882,🕊️ And here the Telegram will fly: a backdoor ...,{'normalized_message': 'and here the telegram ...
2,HackingBlogsGroup.json,41,2024-07-14,2024-07-14T07:31:48+00:00,1720942308,**Automation with SSRFmap👁‍🗨**\n\nTime-tested ...,{'normalized_message': 'automation with ssrfma...
3,HackingBlogsGroup.json,51,2024-07-17,2024-07-17T15:30:33+00:00,1721230233,😡 [Government can read you personal Chats.](h...,{'normalized_message': 'government can read yo...
4,HackingBlogsGroup.json,58,2024-07-17,2024-07-17T17:37:49+00:00,1721237869,Is their more on what exactly is the vulnerabi...,{'normalized_message': 'is their more on what ...


In [7]:
def _load_existing_results() -> tuple[dict[str, list[dict[str, Any]]], set[tuple[str, str]]]:
    existing_by_file: dict[str, list[dict[str, Any]]] = {}
    seen_keys: set[tuple[str, str]] = set()

    for output_path in OUTPUT_DIR.glob("*.json"):
        try:
            with output_path.open("r", encoding="utf-8") as file:
                payload = json.load(file)
        except Exception:
            continue

        if not isinstance(payload, list):
            continue

        file_name = output_path.name
        existing_by_file[file_name] = payload

        for item in payload:
            if not isinstance(item, dict):
                continue
            key = _record_key({"source_file": file_name, **item})
            if key is not None:
                seen_keys.add(key)

    return existing_by_file, seen_keys


def _persist_results_for_file(file_name: str, items: list[dict[str, Any]]) -> None:
    output_path = OUTPUT_DIR / file_name
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(items, file, ensure_ascii=False, indent=2)
        file.write("\n")


def _record_key(post: dict[str, Any]) -> tuple[str, str] | None:
    source_file = post.get("source_file")
    if source_file is None:
        return None

    telegram_id = post.get("_id")
    if telegram_id is not None:
        return str(source_file), f"telegram:{telegram_id}"

    parsed = parse_post_datetime(post)
    message = post.get("message")
    if parsed is not None and isinstance(message, str):
        return str(source_file), f"time_message:{parsed.isoformat()}:{message}"

    post_id = post.get("id")
    if post_id is not None:
        return str(source_file), f"uuid:{post_id}"

    return None


def classify_post(
    post: dict[str, Any],
    ner_entities: list[dict[str, str]] | None = None,
) -> dict[str, Any]:
    enriched = ensure_temporal_fields(post)
    enriched["classification"] = normalize_deterministic_result(enriched, ner_entities=ner_entities)
    return enriched


def _merge_results(by_file: dict[str, list[dict[str, Any]]]) -> list[dict[str, Any]]:
    merged_results: list[dict[str, Any]] = []
    for items in by_file.values():
        merged_results.extend(items)
    return merged_results


def _persist_all_results(by_file: dict[str, list[dict[str, Any]]]) -> None:
    for file_name, items in by_file.items():
        _persist_results_for_file(file_name, items)


def classify_all(
    records: list[dict[str, Any]],
    *,
    n_process: int = NER_N_PROCESS,
    batch_size: int = NER_BATCH_SIZE,
    force_reclassify: bool = True,
) -> list[dict[str, Any]]:
    if force_reclassify:
        by_file: dict[str, list[dict[str, Any]]] = {}
        seen_keys: set[tuple[str, str]] = set()
    else:
        by_file, seen_keys = _load_existing_results()
    to_process: list[dict[str, Any]] = []

    for post in records:
        key = _record_key(post)
        if key is not None and key in seen_keys:
            continue
        to_process.append(post)

    if not to_process:
        print("Nenhum novo post para classificar (todos ja processados no output).")
        return _merge_results(by_file)

    analysis_texts = [_post_text_fields(post)[2] for post in to_process]
    print(f"Processando {len(to_process)} posts com NER n_process={n_process}, batch_size={batch_size}")

    processed_count = 0
    docs = NER.pipe(analysis_texts, batch_size=batch_size, n_process=n_process)
    for post, analysis_text, doc in tqdm(zip(to_process, analysis_texts, docs), total=len(to_process)):
        ner_entities = extract_ner_entities_from_doc(analysis_text, doc)
        classified = classify_post(post, ner_entities=ner_entities)
        source_file = str(classified.get("source_file", "unknown.json"))
        by_file.setdefault(source_file, []).append(classified)

        key = _record_key(classified)
        if key is not None:
            seen_keys.add(key)

        processed_count += 1

    _persist_all_results(by_file)
    merged_results = _merge_results(by_file)

    print(f"Classificados agora: {processed_count} | Total no output: {len(merged_results)}")
    return merged_results

In [8]:
results = classify_all(records, force_reclassify=True)

by_file: dict[str, list[dict[str, Any]]] = {}
for item in results:
    by_file.setdefault(item["source_file"], []).append(item)

print(f"Done. {len(results)} posts disponiveis em {len(by_file)} arquivos no output.")

Processando 6254 posts com NER n_process=16, batch_size=256


100%|██████████| 6254/6254 [00:11<00:00, 544.25it/s]


Classificados agora: 6254 | Total no output: 6254
Done. 6254 posts disponiveis em 9 arquivos no output.


In [9]:
# import os
# import time

# # Optional: add a small delay so you can see the output before shutdown starts
# print("Processing finished. Shutting down Windows in 30 seconds...")
# time.sleep(30)

# # Call Windows shutdown command
# # /s = shutdown, /t 0 = immediately (change to 30 for 30-second delay, etc.)
# os.system("shutdown.exe /s /t 0 /c \"Jupyter notebook requested shutdown\"")

# print("Shutdown command sent.")